# Who Goes There: look at the data on Kaggle

A small notebook for looking at CCT20 photos (item 8.07): a grid of training photos per class,
day vs night, from the attached 512 px cache, and the five data charts of
`trailcam/data/datavis.py` with the night shares of the kept images, drawn with the cache's
day/night sidecar. Section 4 (item 8.08) is an eye check of arm B's box-guided crops: boxed
training photos with their boxes outlined, beside crops drawn by the real arm B sampler and, for
contrast, arm A's. It trains nothing and needs no GPU.

**Never commit this notebook with its outputs.** The photo grid and the crop check show dataset
images: they stay in the Kaggle session. Before a commit, Edit > Clear All Outputs (the data guard refuses a
notebook with any output, R04). Raw dataset images never go into `docs/figures/`; only the
charts of section 3, which are drawn from the metadata and hold no photo, may be copied there.

Only the train split's photos are shown: val, trans_val, cis_test and trans_test are not looked
at here (test-set isolation, R17).

## How to run on Kaggle

1. File > Import Notebook with its GitHub link
   (https://github.com/codx999/who-goes-there/blob/main/notebooks/kaggle_look_at_data.ipynb).
2. Add Input: the private 512 px cache dataset of item 6.03 (`cct20-512`). Accelerator: None.
   Internet: on (the install).
3. Set `COMMIT` to a `main` commit that holds `trailcam/data/datavis.py`, then run the cells one
   by one.
4. Section 3's PNGs in `/kaggle/working/figures/` should match the committed charts in
   `docs/figures/` (same metadata, sidecar and seed); nothing needs downloading.
5. Clear All Outputs before saving a version or committing.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: set COMMIT to a main commit that holds trailcam/data/datavis.py.
from pathlib import Path

COMMIT = "main"  # a full sha pins the code; "main" takes the latest
DATA_ROOT = Path("/kaggle/input/datasets/codx888/cct20-512/cct20_512")  # the 6.03 dataset
REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
SPLIT_SEED = 42  # the split of the data card and config.yaml
PER_CELL = 4  # photos per class for day and for night
FIGURES = Path("/kaggle/working/figures")  # section 3's charts
ARM_B_CROPS = 3  # section 4: arm B crops per boxed photo
ARM_A_CROPS = 2  # and arm A crops, for contrast

## 1. Install `trailcam` and load the metadata

The notebook exception of `CLAUDE.md`: `pip install` of `trailcam` from the public repo. The
metadata is loaded with the sidecar's night flags and split with `SPLIT_SEED`; counts only.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", f"git+{REPO_URL}@{COMMIT}"], check=True
)
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata  # noqa: E402
from trailcam.data.split import make_split  # noqa: E402

meta = load_metadata(DATA_ROOT / "annotations", DATA_ROOT / DAY_NIGHT_SIDECAR)
split = make_split(meta, SPLIT_SEED)
train = split.records["train"]
print(split)
print(f"train: {len(train)} images, {sum(bool(r.night) for r in train)} night")

## 2. Look at the data: training photos per class, day vs night

One row per class: `PER_CELL` day photos, then `PER_CELL` night (infrared) photos, drawn at random
with `SPLIT_SEED`. Titles show the class and day or night only, never a file name. These outputs
are for your eyes in this session: clear them before any commit.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

from trailcam.data.classes import CLASSES

rng = random.Random(SPLIT_SEED)
fig, axes = plt.subplots(
    len(CLASSES), 2 * PER_CELL, figsize=(2.0 * 2 * PER_CELL, 1.6 * len(CLASSES))
)
for row, name in enumerate(CLASSES):
    for col_offset, night in ((0, False), (PER_CELL, True)):
        pool = [r for r in train if r.label == row and r.night is night]
        picks = rng.sample(pool, min(PER_CELL, len(pool)))
        for col in range(PER_CELL):
            ax = axes[row][col_offset + col]
            ax.axis("off")
            if col < len(picks):
                with Image.open(DATA_ROOT / "images" / picks[col].file_name) as image:
                    ax.imshow(image.convert("RGB"))
                ax.set_title(f"{name}, {'night' if night else 'day'}", fontsize=8)
fig.suptitle("CCT20 train photos per class, day vs night (Beery et al. 2018, CDLA-Permissive 1.0)")
fig.tight_layout()
plt.show()

## 3. The data charts with the kept-image night shares

`python -m trailcam.data.datavis` with the sidecar: the same five charts as `docs/figures/`, but
the day/night chart counts the kept images of each split (train, val, trans_val, cis_test,
trans_test) instead of the cache build's numbers over all annotated images. The charts are drawn
from the metadata only, so they may be copied into `docs/figures/` (within the guard's budget).

In [ ]:
from trailcam.data import datavis

datavis.main(
    [
        "--root",
        str(DATA_ROOT / "annotations"),
        "--seed",
        str(SPLIT_SEED),
        "--sidecar",
        str(DATA_ROOT / DAY_NIGHT_SIDECAR),
        "--out",
        str(FIGURES),
    ]
)

## 4. Eye check of arm B crops: boxed training photos, arm B's crops and arm A's

One boxed training photo per class, day and night by turns where the class has both, drawn at
random with `SPLIT_SEED`: the photo with its boxes outlined, then `ARM_B_CROPS` crops from arm
B's box path and `ARM_A_CROPS` from arm A's sampler, with the boxes outlined where they fall. The
sampler is the one training builds, `TrainTransform("B").sampler`: the `BoxCropSampler` of
`trailcam/data/boxcrop.py`, whose `box_crop` is the box path and whose `control` is arm A's
sampler. In training half the boxed images take the box path (`BOX_P`); here every arm B crop
does, so the box path is what you see. A crop is shown at its own shape; training then resizes it
to 224 x 224, flips it and jitters its colours. A crop's title gives the fill, the box's area over
the crop's, for the box it holds whole (the largest such fill), or "box cut" when it holds none
whole: `docs/plan.md` section 6 asks arm B for the whole box at a fill of 20% to 100%. A photo's
title shows its class and day or night only, never a file name. What to look for: every arm B
crop holds its animal whole, not always centred, at fills across the range, on night frames too;
arm A crops often show the animal small or cut it. The line printed under the grid counts the arm
B crops inside that range.

**These outputs are for your eyes in this session only.** They show dataset photos: clear them
(Edit > Clear All Outputs) before saving a version or committing, and never put any of them into
`docs/figures/` (R03, R04). This section changes no training code: it calls the samplers as
training does and draws what they return.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

from trailcam.data.boxcrop import FILL, to_pixels
from trailcam.data.classes import CLASSES
from trailcam.data.dataset import CCT20Dataset, TrainTransform

crop_rng = random.Random(SPLIT_SEED)
picked = []
for label in range(len(CLASSES)):
    boxed = [r for r in train if r.label == label and r.boxes]
    want_night = label % 2 == 1  # day and night by turns, where the class has both
    pool = [r for r in boxed if r.night is want_night] or boxed
    if pool:
        picked.append(crop_rng.choice(pool))
transform = TrainTransform("B")  # arm B's training transform
sampler = transform.sampler  # its BoxCropSampler: box_crop the box path, control arm A's sampler
photos = CCT20Dataset(picked, DATA_ROOT / "images", transform)
torch.manual_seed(SPLIT_SEED)  # both samplers draw from torch's generator
columns = 1 + ARM_B_CROPS + ARM_A_CROPS
fig, axes = plt.subplots(
    len(picked),
    columns,
    figsize=(2.3 * columns, 2.1 * len(picked)),
    squeeze=False,
    layout="constrained",
)
fills = []  # of the arm B crops: the largest fill of a box held whole, 0 if none is
for row, record in enumerate(picked):
    image = photos.load_image(row)
    width, height = image.size
    boxes = [to_pixels(box, width, height) for box in record.boxes]
    crops = [("arm B", sampler.box_crop(width, height, record.boxes)) for _ in range(ARM_B_CROPS)]
    crops += [("arm A", sampler.control(image, record.boxes)) for _ in range(ARM_A_CROPS)]
    for col in range(columns):
        ax = axes[row][col]
        ax.axis("off")
        if col == 0:
            top, left, h, w = 0, 0, height, width
            title = f"{CLASSES[record.label]}, {'night' if record.night else 'day'}"
        else:
            arm, crop = crops[col - 1]
            if crop is None:
                ax.set_title(f"{arm}, no usable box", fontsize=8)
                continue
            top, left, h, w = crop
            whole = [
                (x1 - x0) * (y1 - y0) / (w * h)
                for x0, y0, x1, y1 in boxes
                if left <= max(x0, 0)
                and top <= max(y0, 0)
                and min(x1, width) <= left + w
                and min(y1, height) <= top + h
            ]
            title = f"{arm}, fill {max(whole):.0%}" if whole else f"{arm}, box cut"
            if arm == "arm B":
                fills.append(max(whole, default=0.0))
        ax.imshow(image.crop((left, top, left + w, top + h)))
        for x0, y0, x1, y1 in boxes:
            corner = (x0 - left, y0 - top)
            ax.add_patch(
                Rectangle(corner, x1 - x0, y1 - y0, fill=False, edgecolor="#eb6834", linewidth=1.2)
            )
        ax.set_title(title, fontsize=8)
fig.suptitle(
    "Arm B box crops and arm A crops, CCT20 train photos (Beery et al. 2018, CDLA-Permissive 1.0)"
)
plt.show()
inside = sum(FILL[0] <= fill <= FILL[1] for fill in fills)
print(
    f"arm B: {len(fills)} crops of {len(picked)} photos, {inside} with a whole box at a fill of "
    f"{FILL[0]:.0%} to {FILL[1]:.0%}; fills {min(fills, default=0):.0%} to "
    f"{max(fills, default=0):.0%}"
)